# Notebook 01: Processed Smart Grid Data Exploration

This notebook turns the earlier static `docs/dataset_visualization.png` smoke test into a **tracked, reproducible exploration notebook**. It reads the processed Smart Grid CSVs under `data/processed/`, recreates the core six-panel overview, and writes stable outputs to `results/metrics/` and `results/figures/`.

What this notebook is for:
- sanity-check the processed Smart Grid data before benchmark analysis
- give teammates a canonical first stop for understanding dataset shape and fault/health patterns
- provide a reproducible replacement for the older image-only artifact from issue `#103`

What this notebook is not for:
- benchmark latency analysis (`Notebook 02`)
- orchestration comparison (`Notebook 03`)
- paper-ready generated-vs-manual scenario analysis (`Notebook 04`)


## Environment and run expectations

- Expected to run from anywhere inside the repo; the notebook auto-discovers the repo root.
- Reads only repo-tracked inputs from `data/processed/`.
- Writes reproducible outputs to:
  - `results/metrics/notebook01_asset_tier_summary.csv`
  - `results/metrics/notebook01_fault_counts_by_tier.csv`
  - `results/figures/notebook01_dataset_overview.png`
  - `results/figures/notebook01_dataset_overview.pdf`
- Package versions are captured below for reproducibility.


In [ ]:
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.options.display.max_columns = 20
pd.options.display.float_format = lambda x: f"{x:,.2f}"


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate repo root from current working directory.")


ROOT = find_repo_root(Path.cwd().resolve())
DATA_DIR = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root: {ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"pandas: {pd.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"Platform: {platform.platform()}")


## Load the processed datasets

In [ ]:
HI_FULL_HEALTH_DAYS = 1093.0
EXPECTED_DATASETS = {
    "asset_metadata": DATA_DIR / "asset_metadata.csv",
    "dga_records": DATA_DIR / "dga_records.csv",
    "failure_modes": DATA_DIR / "failure_modes.csv",
    "fault_records": DATA_DIR / "fault_records.csv",
    "rul_labels": DATA_DIR / "rul_labels.csv",
    "sensor_readings": DATA_DIR / "sensor_readings.csv",
}

missing_files = [path.relative_to(ROOT) for path in EXPECTED_DATASETS.values() if not path.exists()]
if missing_files:
    formatted = "\n".join(f"- {path}" for path in missing_files)
    raise RuntimeError(f"Missing expected processed data files:\n{formatted}")

asset_metadata = pd.read_csv(EXPECTED_DATASETS["asset_metadata"])
dga_records = pd.read_csv(EXPECTED_DATASETS["dga_records"], parse_dates=["sample_date"])
failure_modes = pd.read_csv(EXPECTED_DATASETS["failure_modes"])
fault_records = pd.read_csv(EXPECTED_DATASETS["fault_records"])
rul_labels = pd.read_csv(EXPECTED_DATASETS["rul_labels"], parse_dates=["timestamp"])
sensor_readings = pd.read_csv(EXPECTED_DATASETS["sensor_readings"], parse_dates=["timestamp"])


def assign_tier(row: pd.Series) -> str:
    if row["fdd_category"] == 1 and row["rul_days"] >= HI_FULL_HEALTH_DAYS:
        return "healthy_long"
    if row["fdd_category"] == 1:
        return "healthy_aging"
    if row["fdd_category"] in [2, 3]:
        return "minor_fault"
    return "serious_fault"


TIER_ORDER = ["healthy_long", "healthy_aging", "minor_fault", "serious_fault"]
TIER_LABELS = {
    "healthy_long": "Healthy / long RUL",
    "healthy_aging": "Healthy / aging",
    "minor_fault": "Minor or moderate fault",
    "serious_fault": "Serious fault",
}

asset_metadata["tier"] = asset_metadata.apply(assign_tier, axis=1)
dga_records = dga_records.merge(asset_metadata[["transformer_id", "tier"]], on="transformer_id", how="left")
fault_records = fault_records.merge(asset_metadata[["transformer_id", "tier"]], on="transformer_id", how="left")
rul_labels = rul_labels.merge(asset_metadata[["transformer_id", "tier"]], on="transformer_id", how="left")

summary = pd.DataFrame([
    {"dataset": "asset_metadata", "rows": len(asset_metadata), "columns": asset_metadata.shape[1]},
    {"dataset": "dga_records", "rows": len(dga_records), "columns": dga_records.shape[1]},
    {"dataset": "failure_modes", "rows": len(failure_modes), "columns": failure_modes.shape[1]},
    {"dataset": "fault_records", "rows": len(fault_records), "columns": fault_records.shape[1]},
    {"dataset": "rul_labels", "rows": len(rul_labels), "columns": rul_labels.shape[1]},
    {"dataset": "sensor_readings", "rows": len(sensor_readings), "columns": sensor_readings.shape[1]},
])
summary


## Quick inventory checks

In [ ]:
asset_tier_summary = (
    asset_metadata.groupby("tier", observed=False)
    .agg(
        transformer_count=("transformer_id", "count"),
        mean_age_years=("age_years", "mean"),
        mean_rul_days=("rul_days", "mean"),
        min_rul_days=("rul_days", "min"),
        max_rul_days=("rul_days", "max"),
    )
    .reindex(TIER_ORDER)
    .rename(index=TIER_LABELS)
)
asset_tier_summary["transformer_count"] = asset_tier_summary["transformer_count"].fillna(0).astype(int)
asset_tier_summary.to_csv(METRICS_DIR / "notebook01_asset_tier_summary.csv")
asset_tier_summary


In [ ]:
fault_counts_by_tier = (
    pd.crosstab(fault_records["tier"], fault_records["fault_type"])
    .reindex(TIER_ORDER)
    .rename(index=TIER_LABELS)
    .fillna(0)
    .astype(int)
)
fault_counts_by_tier.to_csv(METRICS_DIR / "notebook01_fault_counts_by_tier.csv")
fault_counts_by_tier


## Recreate the six-panel dataset smoke test

This mirrors the spirit of the earlier static image while making every panel derivable from repo-tracked data.


In [ ]:
representatives = (
    asset_metadata.sort_values(["tier", "transformer_id"])
    .groupby("tier", observed=False)
    .head(1)
    [["tier", "transformer_id"]]
)
rep_ids = representatives["transformer_id"].tolist()
rep_map = {row.tier: row.transformer_id for row in representatives.itertuples(index=False)}

temp_timeseries = sensor_readings[
    (sensor_readings["sensor_id"] == "winding_temp_top_c")
    & (sensor_readings["transformer_id"].isin(rep_ids))
].merge(asset_metadata[["transformer_id", "tier"]], on="transformer_id", how="left")

rul_window = rul_labels[rul_labels["transformer_id"].isin(rep_ids)].copy()
latest_rul = asset_metadata[["transformer_id", "tier", "rul_days"]].copy()

c2h2_view = dga_records[["transformer_id", "tier", "dissolved_c2h2_ppm"]].copy()
dga_snapshot = (
    dga_records.melt(
        id_vars=["transformer_id", "tier"],
        value_vars=[
            "dissolved_h2_ppm",
            "dissolved_ch4_ppm",
            "dissolved_c2h2_ppm",
            "dissolved_c2h4_ppm",
            "dissolved_c2h6_ppm",
        ],
        var_name="gas",
        value_name="ppm",
    )
    .groupby(["tier", "gas"], observed=False)["ppm"]
    .median()
    .reset_index()
)
dga_snapshot["tier"] = pd.Categorical(dga_snapshot["tier"], categories=TIER_ORDER, ordered=True)
dga_heatmap = (
    dga_snapshot.pivot(index="tier", columns="gas", values="ppm")
    .reindex(TIER_ORDER)
    .rename(index=TIER_LABELS)
)

fault_heatmap = fault_counts_by_tier.copy()

fig, axes = plt.subplots(3, 2, figsize=(16, 18), constrained_layout=True)

# 1. Representative sensor time-series view
ax = axes[0, 0]
for tier in TIER_ORDER:
    subset = temp_timeseries[temp_timeseries["tier"] == tier].sort_values("timestamp")
    if subset.empty:
        continue
    ax.plot(subset["timestamp"], subset["value"], label=f"{TIER_LABELS[tier]} ({rep_map[tier]})", linewidth=2)
ax.set_title("Winding temperature over time (representative transformer per tier)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Top winding temp (°C)")
handles, labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(fontsize=8, loc="upper left")

# 2. C2H2 arcing indicator
ax = axes[0, 1]
sns.boxplot(data=c2h2_view, x="tier", y="dissolved_c2h2_ppm", order=TIER_ORDER, ax=ax)
sns.stripplot(data=c2h2_view, x="tier", y="dissolved_c2h2_ppm", order=TIER_ORDER, ax=ax, color="black", alpha=0.65, size=5)
ax.set_title("C2H2 arcing indicator by tier")
ax.set_xlabel("Tier")
ax.set_ylabel("Dissolved C2H2 (ppm)")
ax.set_xticks(range(len(TIER_ORDER)), labels=[TIER_LABELS[t] for t in TIER_ORDER], rotation=20, ha="right")

# 3. RUL window
ax = axes[1, 0]
for tier in TIER_ORDER:
    subset = rul_window[rul_window["tier"] == tier].sort_values("timestamp")
    if subset.empty:
        continue
    ax.plot(subset["timestamp"], subset["rul_days"], label=f"{TIER_LABELS[tier]} ({rep_map[tier]})", linewidth=2)
ax.set_title("RUL window (representative transformer per tier)")
ax.set_xlabel("Timestamp")
ax.set_ylabel("RUL (days)")
ax.legend(fontsize=8, loc="upper right")

# 4. RUL distribution by tier
ax = axes[1, 1]
sns.boxplot(data=latest_rul, x="tier", y="rul_days", order=TIER_ORDER, ax=ax)
sns.stripplot(data=latest_rul, x="tier", y="rul_days", order=TIER_ORDER, ax=ax, color="black", alpha=0.65, size=5)
ax.set_title("RUL distribution by tier")
ax.set_xlabel("Tier")
ax.set_ylabel("RUL (days)")
ax.set_xticks(range(len(TIER_ORDER)), labels=[TIER_LABELS[t] for t in TIER_ORDER], rotation=20, ha="right")

# 5. DGA gas snapshot
ax = axes[2, 0]
sns.heatmap(dga_heatmap, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax)
ax.set_title("Median DGA gas snapshot by tier")
ax.set_xlabel("Gas channel")
ax.set_ylabel("Tier")

# 6. Fault records by tier
ax = axes[2, 1]
sns.heatmap(fault_heatmap, annot=True, fmt="d", cmap="Blues", ax=ax)
ax.set_title("Fault records by tier")
ax.set_xlabel("Fault type")
ax.set_ylabel("Tier")

figure_png = FIGURES_DIR / "notebook01_dataset_overview.png"
figure_pdf = FIGURES_DIR / "notebook01_dataset_overview.pdf"
fig.savefig(figure_png, dpi=200, bbox_inches="tight")
fig.savefig(figure_pdf, bbox_inches="tight")
plt.show()

print(f"Saved figure: {figure_png.relative_to(ROOT)}")
print(f"Saved figure: {figure_pdf.relative_to(ROOT)}")


## Observations from the processed data

A few stable takeaways already show up before any benchmark runs:

- The processed data cleanly separates the three populated health tiers in both remaining useful life and dissolved-gas behavior.
- The `healthy_aging` tier is currently empty under `HI_FULL_HEALTH_DAYS = 1093.0`; every `fdd_category = 1` transformer in the current processed CSVs has `rul_days >= 2526`, so that middle bucket is still a planned category rather than an observed cluster.
- The processed data is rich enough to support multi-domain maintenance scenarios: asset metadata, DGA evidence, time-series sensor readings, RUL labels, and fault-history records all line up on `transformer_id`.
- The earlier `dataset_visualization.png` was directionally useful, but this notebook is the stronger artifact because a teammate can now rerun it against the tracked CSVs and inspect the exact transformations.
- This notebook should remain focused on **data understanding**. Once we switch to benchmark outputs under `benchmarks/`, that belongs to Notebook 02 and later analysis notebooks.


In [ ]:
print("Generated outputs:")
for path in [
    METRICS_DIR / "notebook01_asset_tier_summary.csv",
    METRICS_DIR / "notebook01_fault_counts_by_tier.csv",
    FIGURES_DIR / "notebook01_dataset_overview.png",
    FIGURES_DIR / "notebook01_dataset_overview.pdf",
]:
    print("-", path.relative_to(ROOT))
